<p style="text-align:right">Фазлыев А.А, МОиАИС 20.01-1</p>
<h1 style="text-align:center">Лабораторная работа №4</h1>
<h2 style="text-align:center">Тема: Приближение функций. Построение интерполяционного многочлена в форме Лагранжа и в форме Ньютона</h2>

### 1. Постановка задачи

Приближение (аппроксимация) функции - $f(x) \approx F(x)$

Отклонение $f(x)$ от $F(x)$ (в некотором смысле) должно быть наименьшим в заданной области

Аппроксимация называется точечной, если аппроксимирующая функция $F(x)$ строится на дискретном множестве точек $X\{x_0, x_1, \dots, x_n\}$

#### Интерполирование
При интерполированании, под близостью $f(x) \approx F(X)$ подразумевается, что функции $f(x)$ и $F(x)$ совпадают на дискретном множестве точек $x_i (i = 0, 1, 2, \dots, n)$, на котором задана исходная функция $f(x)$ (но могут отличаться при $x \neq x_i$)

$F(x_i) = f(x_i), i = 0, 1, 2, \dots, n$

Обычно точки называют узлами
интерполяции, а функцию $F(x)$ , если она ищется в виде
многочлена - *интерполяционным многочленом*

#### Интерполяционный многочлен Лагранжа 
$F_n(x) = L_n(x) = \sum\limits_{i=0}^n f(x_i) \prod\limits_{\substack{j \neq i \\ j = 0}}^n \frac{(x - x_j)}{(x_i - x_j)}$


#### Оценка погрешности метода
$|R_n(x)| = |f(x) - L_n(x)| \leq \frac{M_{n+1}}{(n + 1)!}|\Pi_{n+1}|$

$\Pi_{n + 1} = (x - x_0)(x - x_1)\dots(x - x_n)$
$M_{n + 1} \underset{a \leq x \leq b}{\max}f^{n + 1}(x)$

#### Интерполяционный многочлен Ньютона
$P_n(x) = f(x_0) + f(x_0, x_1)(x - x_0) + f(x0, x1, x2)(x - x_0)(x - x1) + \dots + f(x_0, \dots, x_1)(x - x_0)\dots(x - x_n)$

##### Разделенная разность n-ого порядка
\begin{cases}
f(x_0; x_1; \dots; x_n) = \frac{f(x_1; \dots; x_n) - f(x_0; \dots; x_{n-1})}{x_n - x_0} \\
f(x_0) = y_0
\end{cases}


#### Порядок выполнения работы
Многократно дифференцируемая функция $y=f(x)$ задана таблицей значений

| **$x$** | $x_0$ | $x_1$ | $\dots$ | $x_n$ |
|---------|-------|-------|---------|-------|
| **$y$** | $y_0$ | $y_1$ | $\dots$ | $y_n$ |

Выбрать отрезок $[a,b]$, разбить его с шагом $h=b-an$, ($n$ - число интервалов разбиения), ввести сетку $x_i=x_0+ih, i=0, 1,\dots,n$ и составить таблицу $\{x_i,f(x_i)\}$

Построить по заданной таблице значений:
 + интерполяционный многочлен Лагранжа $L_n(x)$;
 + первый многочлен Ньютона $P_n(x)$;
 + вывести теоретическую оценку погрешности интерполяции $R_n(x)$
Результаты представить в виде таблицы:



|$\tilde x_i = x_0 + i\frac{h}{2}$|$f(\tilde x_i)$|$Ln(\tilde x_i)$|$Pn(\tilde x_i)$|$|f(\tilde x_i) - Ln(\tilde x_i)|$|$Rn(\tilde x_i)$|
|---------------------------------|---------------|----------------|----------------|-----------------------------------|------------------|
|$\cdots$                         |$\cdots$       |$\cdots$        |$\cdots$        |$\cdots$|$\cdots$                   |    


### 2. Метод решения

In [1]:
import numpy as np
from scipy.misc import derivative
from ipywidgets import interact, Dropdown, FloatRangeSlider, BoundedIntText
from matplotlib import pyplot as plt

интерполяционный многочлен Лагранжа

In [2]:
def lagrange(x, y, x0):
    n = len(x)
    p = 0
    for i in range(n):
        prod = y[i]
        for j in range(n):
            if j != i:
                prod *= (x0 - x[j]) / (x[i] - x[j])
        p += prod
    return p

первый многочлен Ньютона

In [3]:
def newton(x, y, x0):
    n = len(x)
    a = np.zeros((n, n))
    
    a[0] = y
    for i in range(1, n):
        for j in range(i, n):
            a[i, j] = (a[i - 1, j] - a[i - 1, j - 1]) / (x[j] - x[j - i])
    
    result = a[0,0]
    for i in range(1, n):
        p = a[i, i]
        for j in range(i):
            p *= (x0 - x[j])
        result += p
    return result


Оценка ошибки

In [4]:
def error(M, x, x0):
    fact = 1
    p = 1
    for i in range(len(x)):
        p *= (x[i] - x0)
        fact *= i + 1
    return M / fact * abs(p)

In [5]:
@interact(
    f=Dropdown(
        options={
            'cos': np.cos, 
            'x^2': np.square,
            'x^3': lambda x: np.power(x, 3),
            'exp': np.exp,
            
        },
        description='$f(x)$:',
    ),
    a_b=FloatRangeSlider(
        value=[-5, 5],
        min=-10,
        max=10,
        step=0.1,
        description='$[a, b]$:',
    ),
    n=BoundedIntText(
        min=2,
        step=1,
        description='$n$',
    )
)
def show_plot(f, a_b, n):
    a, b = a_b
    x = np.linspace(a, b, n)
    y = f(x)

    x_t = np.linspace(a, b, n * 2)
    y_t = f(x_t)
    y_lag = lagrange(x, y, x_t) 
    
    M = derivative(f, np.linspace(a, b, 100), n=len(x), order=101).max()
    err = error(M, x, x_t)
    
    fig1, ax = plt.subplot_mosaic("AB", figsize=(10, 5))
    
    
    ln = np.linspace(a, b, 200)
    fln = f(ln)
    
    ax["A"].plot(ln, fln)
    ax["A"].set_title('интерполяционный многочлен Лагранжа')
    ax["A"].plot(x_t, lagrange(x, y, x_t), marker='x')
    ax["A"].fill_between(
        x_t, f(x_t) - err, f(x_t) + err, alpha=0.2
    )
    ax["A"].set_xlim([a, b])
    ax["A"].set_ylim([fln.min(), fln.max()])
    
    
    ax["B"].set_title('первый многочлен Ньютона')
    ax["B"].plot(ln, fln)
    ax["B"].plot(x_t, newton(x, y, x_t), marker='x', color='green')
    ax["B"].fill_between(
        x_t, f(x_t) - err, f(x_t) + err, alpha=0.2
    )
    ax["B"].set_xlim([a, b])
    ax["B"].set_ylim([fln.min(), fln.max()])
    fig1.tight_layout()

    
    
    fig2, ax = plt.subplots(figsize=(10, 5))
    ax.table(
        cellText=np.transpose([
            x_t, y_t, lagrange(x, y, x_t), newton(x, y, x_t), abs(y_t - y_lag), err
        ]).round(3), 
        colLabels=[
            r'$\tilde x_i = x_0 + i\frac{h}{2}$',
            r'$f(\tilde x_i)$',
            r'$Ln(\tilde x_i)$',
            r'$Pn(\tilde x_i)$',
            r'$|f(\tilde x_i) - Ln(\tilde x_i)|$',
            r'$Rn(\tilde x_i)$'
        ],
        loc="top"
    )
    ax.axis("off")
    
    fig1, fig2

interactive(children=(Dropdown(description='$f(x)$:', options={'cos': <ufunc 'cos'>, 'x^2': <ufunc 'square'>, …